# Human Development Index (HDI) Prediction
### Machine Learning + Flask End-to-End Project

This notebook walks through the full ML lifecycle to build a **Linear Regression** model that predicts a country's **HDI score** from four indicators:
1. Life expectancy
2. Mean years of schooling
3. Expected years of schooling
4. Gross National Income (GNI) per capita

Workflow covered:
- Epic 2: Import libraries
- Epic 3: Dataset understanding & visualization
- Epic 4: Data preprocessing & label encoding
- Epic 5: Train / Test split
- Epic 6: Fit Linear Regression & evaluate
- Epic 7: Save the model with Pickle

## Epic 2 — Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import pickle

import warnings
warnings.filterwarnings('ignore')

# Display plots inline and set a clean style
%matplotlib inline
sns.set_style('whitegrid')
print('All libraries imported successfully.')

## Epic 3 — Dataset Download and Understanding

### Story 1 & 2: Load the dataset and explore its structure

In [ ]:
# Load the HDI dataset
df = pd.read_csv('../Dataset/HDI.csv')
df.head()

In [ ]:
# Shape of the dataset
print('Rows    :', df.shape[0])
print('Columns :', df.shape[1])

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary of numeric features
df.describe()

### Story 3: Data Visualization

In [ ]:
# Distribution of the target variable (HDI)
plt.figure(figsize=(8, 5))
sns.histplot(df['HDI'], bins=25, kde=True, color='steelblue')
plt.title('Distribution of HDI Scores', fontsize=14)
plt.xlabel('HDI')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise relationships between indicators and HDI
sns.pairplot(df.drop(columns=['Country']))
plt.suptitle('Pairplot of HDI Indicators', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df.drop(columns=['Country']).corr()
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Epic 4 — Data Preprocessing and Label Encoding

### Story 1: Select dependent and independent variables

- **Independent variables (X):** Life expectancy, Mean years of schooling, Expected years of schooling, GNI per capita
- **Dependent variable (y):** HDI

In [ ]:
# Independent features
X = df[['Life expectancy',
        'Mean years of schooling',
        'Expected years of schooling',
        'Gross National Income (GNI) per capita']]

# Target variable
y = df['HDI']

X.head()

### Story 2: Check for missing values

In [ ]:
# Check for nulls across the whole dataset
df.isnull().sum()

### Story 3: Label Encoding

The `Country` column is categorical text. We convert it to numerical labels so it can be used/saved alongside the model. The numeric indicators (already numeric) feed the regression directly.

In [ ]:
# Label-encode the Country column (categorical -> numeric)
le = LabelEncoder()
df['Country_encoded'] = le.fit_transform(df['Country'])

# Preview encoded values
df[['Country', 'Country_encoded']].head(10)

### Story 4: Confirm the cleaned dataset is ready

In [ ]:
print('Final feature matrix shape :', X.shape)
print('Target vector shape        :', y.shape)
df_clean = X.copy()
df_clean['HDI'] = y
df_clean.head()

## Epic 5 — Train / Test Split

Split the data so the model learns on one portion and is evaluated on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training samples :', X_train.shape[0])
print('Testing samples  :', X_test.shape[0])

## Epic 6 — Fitting the Model

### Story 1: Train the Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print('Model trained successfully.')
print('Coefficients :', model.coef_)
print('Intercept    :', model.intercept_)

### Story 2: Generate predictions

In [ ]:
y_pred = model.predict(X_test)

# Compare actual vs predicted
comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred.round(3)})
comparison.head(10)

### Story 3: Evaluate model performance

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error (MAE) : {mae:.4f}')
print(f'Mean Squared Error (MSE)  : {mse:.6f}')
print(f'Root Mean Squared Error   : {rmse:.4f}')
print(f'R-squared (R2) Score      : {r2:.4f}')

In [ ]:
# Visualize actual vs predicted
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, color='teal', alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual HDI')
plt.ylabel('Predicted HDI')
plt.title('Actual vs Predicted HDI')
plt.tight_layout()
plt.show()

## Epic 7 — Saving the Model

### Story 1 & 2: Serialize with Pickle for reuse and deployment

In [ ]:
# Save the trained model so the Flask app can load it without retraining
with open('../Flask/HDI.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Model saved to ../Flask/HDI.pkl')

# Quick sanity-check: reload and predict a sample
with open('../Flask/HDI.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

sample = pd.DataFrame([[82.6, 12.9, 18.2, 66000]],
                      columns=X.columns)
pred = loaded_model.predict(sample)[0]
print(f'Sample prediction (Norway-like): HDI = {pred:.3f}')

---

### ✅ Pipeline complete
The trained Linear Regression model has been saved as `HDI.pkl`.
The next step (Epic 8) is to build the Flask web application that uses this model to serve live HDI predictions to users.